Για τις ανάγκες της προκαταρκτικής αυτής εργασίας έχουν δημιουργηθεί πακέτα κώδικα που χρησιμοποιούνται ως βιβλιοθήκες στη ροή του notebook.

In [2]:
import importlib
import DatasetBuilder
import visualisation_mod
import pandas as pd

# Force a fresh load of both modules
importlib.reload(DatasetBuilder)
importlib.reload(visualisation_mod)

from DatasetBuilder import MultimodalDatasetFactory, MultimodalUniverseFactory
from visualisation_mod import UniverseVisualizer

DATA_ROOT = "/content/drive/MyDrive/Data_science_and_applications"
factory = MultimodalUniverseFactory(base_path=DATA_ROOT)
print("✓ Modules synced and ready.")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/home/thinpan/.local/lib/python3.10/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/home/thinpan/.local/lib/python3.10/site-packages/traitlets/config/application.py", line 1043, in launch_instance
    app.start()
  File "/home/thinpan/.local/lib/python3.10/site-

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

Στο πρώτο αυτό notebook της συνολικής εργασίας για λογαριασμό του μαθήματος «Επιστήμη δεδομένων και εφαρμογές» θα απασχοληθούμε με το εξής ερώτημα:
Πως μπορούμε να προσεγκίσουμε την τιμή μιας μεταβλητής που δεν γνωρίζουμε σε έναν μαθηματικό τύπο που υπολογίζει την θερμοκρασία ενός αστέρα.



η στρατηγική μας θα είναι η εξής:

1. Θα δημιουργήσουμε ένα sub-dataset από το survey Gaia για data αστέρων. μεταβλητες:
* $m$: Apparent magnitude (in Gaia, this is phot_g_mean_mag).
* $p$: Parallax in milliarcseconds (mas).

* The formula for Absolute Magnitude is:$$M = m + 5 + 5 \log_{10}\left(\frac{p}{1000}\right)$$

Ορίζουμε ως εξαρτημένη μεταβλητή την θερμοκρασία των αστέρων.

2. Θα ψάξουμε πτιοκοποιήσουμε την εξαρτημένη μεταβλητή (tempreture) σε ένα διάγραμμα με την  



Part 1 — Exploratory Data Analysis

Πρώτο βήμα στην δημιουργία του sub-dataset ειναι to registration. Δηλώνουμε ποια mods θελουμε να χρησιμοποιήσουμε. Αφού ορίσουμε ποια κομμάτια χρειαζόμαστε, θα κατεβάσουμε με βάση αυτή την λίστα τα επιθυμιτά κομμάτια.

In [ ]:
sample_df = factory.load_as_dataframe("MultimodalUniverse/gaia","train", n_samples=10)

Opening stream to: MultimodalUniverse/gaia...
Fetched 10 items (Sieved 10 total).


In [ ]:
print("Available columns in this dataset:")
print(type(sample_df), sample_df.columns )

Available columns in this dataset:
<class 'pandas.core.frame.DataFrame'> Index(['object_id', 'healpix', 'ra', 'dec', 'coeff', 'coeff_error',
       'phot_g_mean_mag', 'phot_g_mean_flux', 'phot_g_mean_flux_error',
       'phot_bp_mean_mag', 'phot_bp_mean_flux', 'phot_bp_mean_flux_error',
       'phot_rp_mean_mag', 'phot_rp_mean_flux', 'phot_rp_mean_flux_error',
       'phot_bp_rp_excess_factor', 'bp_rp', 'bp_g', 'g_rp', 'ra', 'ra_error',
       'dec', 'dec_error', 'parallax', 'parallax_error', 'pmra', 'pmra_error',
       'pmdec', 'pmdec_error', 'ra_dec_corr', 'ra_parallax_corr',
       'ra_pmra_corr', 'ra_pmdec_corr', 'dec_parallax_corr', 'dec_pmra_corr',
       'dec_pmdec_corr', 'parallax_pmra_corr', 'parallax_pmdec_corr',
       'pmra_pmdec_corr', 'radial_velocity', 'radial_velocity_error',
       'rv_template_fe_h', 'rv_template_logg', 'rv_template_teff',
       'ag_gspphot', 'ag_gspphot_lower', 'ag_gspphot_upper', 'azero_gspphot',
       'azero_gspphot_lower', 'azero_gspphot_upper'

In [ ]:
df = factory.get_streaming_examples("MultimodalUniverse/gaia","train", n_samples=10)


Opening stream to: MultimodalUniverse/gaia...
Fetched 10 items (Sieved 10 total).


In [ ]:
df

AttributeError: 'list' object has no attribute 'shape'

In [ ]:
# Test: Peek at raw data values for the keys we are filtering on
print("--- Gaia Stream Diagnostic ---")
ds = factory.get_streaming_examples("MultimodalUniverse/gaia", split='train')
sample_batch = []
it = iter(ds)

for i in range(3):
    item = next(it)
    print(f"\nItem {i} Data:")
    print(f" > 'parallax': {item.get('parallax')} (Type: {type(item.get('parallax'))})")
    print(f" > 'ruwe': {item.get('ruwe')} (Type: {type(item.get('ruwe'))})")
    print(f" > 'phot_g_mean_mag': {item.get('phot_g_mean_mag')}")
    # Check if the keys we used for Legacy Survey even exist here
    print(f" > 'type' exists?: {'type' in item}")
    print(f" > 'shape_r' exists?: {'shape_r' in item}")

--- Gaia Stream Diagnostic ---
Opening stream to: MultimodalUniverse/gaia...
Fetched 4 items (Sieved 4 total).

Item 0 Data:
 > 'parallax': None (Type: <class 'NoneType'>)
 > 'ruwe': None (Type: <class 'NoneType'>)
 > 'phot_g_mean_mag': None
 > 'type' exists?: False
 > 'shape_r' exists?: False

Item 1 Data:
 > 'parallax': None (Type: <class 'NoneType'>)
 > 'ruwe': None (Type: <class 'NoneType'>)
 > 'phot_g_mean_mag': None
 > 'type' exists?: False
 > 'shape_r' exists?: False

Item 2 Data:
 > 'parallax': None (Type: <class 'NoneType'>)
 > 'ruwe': None (Type: <class 'NoneType'>)
 > 'phot_g_mean_mag': None
 > 'type' exists?: False
 > 'shape_r' exists?: False


In [ ]:
# Check the root keys of the first item
print(f"Root keys in Gaia item: {ds[0].keys()}")

Root keys in Gaia item: dict_keys(['spectral_coefficients', 'photometry', 'astrometry', 'radial_velocity', 'gspphot', 'flags', 'corrections', 'object_id', 'healpix', 'ra', 'dec'])
